# CertifyTube Viva Notebook - XGBoost Pipeline

This notebook is a single-flow demo for viva:
1. Setup
2. Data + feature contract check
3. XGBoost train + evaluate
4. Local inference + SHAP contributors
5. FastAPI endpoint demo (`/engagement/analyze/xgboost`)

# CertifyTube Viva Notebook - XGBoost Pipeline

This notebook is a single-flow demo for viva:
1. Setup
2. Data + feature contract check
3. XGBoost train + evaluate
4. Local inference + SHAP contributors
5. FastAPI endpoint demo (`/engagement/analyze/xgboost`)

In [ ]:
# If running in a fresh Colab runtime, uncomment and run:
# !git clone https://github.com/your-username/certifytube_ml_model.git
# %cd certifytube_ml_model

from pathlib import Path
ROOT = Path.cwd()
print('Working directory:', ROOT)
assert (ROOT / 'app').exists(), 'Run this notebook from project root.'

In [ ]:
# Install project dependencies
%pip install -q -r requirements.txt

In [ ]:
import json
import pandas as pd

DATA_PATH = 'data/processed/sessions_features.csv'
CONTRACT_PATH = 'verification/engagement/contracts/feature_contract_v1.json'

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)

data_preview = df.head(2)
data_preview

In [ ]:
with open(CONTRACT_PATH, 'r', encoding='utf-8') as f:
    contract = json.load(f)

feature_names = contract['features']
print('Feature version:', contract['feature_version'])
print('Total required features:', len(feature_names))
print('First 10 features:', feature_names[:10])

## Train and Evaluate XGBoost
Set `RUN_TRAINING = False` if artifacts are already prepared and you only want fast demo.

In [ ]:
RUN_TRAINING = True
if RUN_TRAINING:
    !python verification/engagement/xgboost/training/train.py

In [ ]:
!python verification/engagement/xgboost/training/evaluate.py

In [ ]:
import json
from pathlib import Path
import pandas as pd

metrics_path = Path('reports/metrics_test.json')
if metrics_path.exists():
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    print('XGBoost test metrics')
    for k, v in metrics.items():
        print(f'{k}: {v}')

sweep_path = Path('reports/threshold_sweep.csv')
if sweep_path.exists():
    display(pd.read_csv(sweep_path).head())

In [ ]:
from verification.engagement.xgboost.inference.predict import predict_engagement
from verification.engagement.xgboost.explain.shap_explain import compute_local_shap, top_contributors

sample_row = df.iloc[0]
feature_payload = {f: float(sample_row[f]) for f in feature_names}

pred = predict_engagement(feature_payload)
shap_rows = compute_local_shap(feature_payload)
neg, pos = top_contributors(shap_rows, k=3)

print('Local prediction:', pred)
print('
Top positive contributors:')
for r in pos:
    print(r)
print('
Top negative contributors:')
for r in neg:
    print(r)

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
payload = {
    'session_id': 'viva-xgb-001',
    'feature_version': 'v1.0',
    'features': feature_payload,
}
response = client.post('/engagement/analyze/xgboost', json=payload)
print('Status:', response.status_code)
print('Keys:', list(response.json().keys()))
response.json()

## Viva Speaking Points (XGBoost)
- Why XGBoost: strong tabular performance for behavioral features.
- Why threshold 0.85: policy choice to reduce false-positive certification.
- Evidence to show: AUC/PR, confusion matrix, threshold sweep.
- Backend readiness: same model used through FastAPI endpoint with strict feature contract.